# Phase 1 (functional): inferential statistics

Inferential tests over the circuit-functional dataframe across 11 hypothesis domains
(D1-D11). Default tests are non-parametric (Kruskal-Wallis, Mann-Whitney, Wilcoxon),
with Jonckheere-Terpstra for ordered bands and Spearman for correlations. FDR
correction (Benjamini-Hochberg) is applied globally; alpha=0.05; bootstrap CIs use
10,000 resamples. Inputs: `full_circuit_data.csv`, `full_transfer_data.csv`,
`band_jaccard.csv`, `random_baseline_results.json`.

## 1. Setup & Data Loading

In [1]:
import sys
import json
import warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path("LSC_circuit_analysis/01_Phase_Functional")
sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.constants import *
from utils.metrics import compute_generalization_gap, compute_asymmetric_transfer
from utils.stats import (
    TestAccumulator,
    safe_kruskal,
    safe_mannwhitneyu,
    safe_wilcoxon,
    safe_spearmanr,
    jonckheere_terpstra,
    cohens_d,
    cohens_d_paired,
    rank_biserial,
    eta_squared,
    bootstrap_ci,
    bootstrap_ci_diff,
)
from utils.plotting import setup_plotting, save_figure

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()
np.random.seed(RANDOM_SEED)

In [2]:
df_circuit = pd.read_csv(ANALYSIS_DIR / "full_circuit_data.csv")
df_transfer = pd.read_csv(ANALYSIS_DIR / "full_transfer_data.csv")
print(f"Circuit data: {df_circuit.shape}")
print(f"Transfer data: {df_transfer.shape}")

acc = TestAccumulator()

Circuit data: (75, 27)
Transfer data: (375, 18)


## 2. Domain 1: base model band effects (B1-B4)

B1: Kruskal-Wallis on `base_accuracy` across bands. B2: Jonckheere-Terpstra trend test. B3, B4: KW on `base_top5_accuracy` and `base_mean_correct_prob`.

In [3]:
for model in MODELS:
    md = df_circuit[df_circuit["model"] == model]

    # B1: Base accuracy differs by band (KW, all 5 bands)
    groups = [md[md["band"] == b]["base_accuracy"].dropna().values for b in BANDS]
    H, p = safe_kruskal(*groups)
    acc.add_test(
        "D1_Base",
        "B1",
        model,
        "Kruskal-Wallis",
        "base_acc_by_band",
        H,
        p,
        eta_squared(groups),
        "eta_squared",
        n1=sum(len(g) for g in groups),
    )

    # B2: Monotonic increase with frequency (JT, frequency bands only)
    freq_groups = [
        md[md["band"] == b]["base_accuracy"].dropna().values for b in FREQUENCY_BANDS
    ]
    Z, p_jt = jonckheere_terpstra(freq_groups, alternative="increasing")
    acc.add_test(
        "D1_Base", "B2", model, "Jonckheere-Terpstra", "base_acc_trend", Z, p_jt, Z, "Z"
    )

    # B3: Top-5 accuracy differs by band
    groups_t5 = [
        md[md["band"] == b]["base_top5_accuracy"].dropna().values for b in BANDS
    ]
    H, p = safe_kruskal(*groups_t5)
    acc.add_test(
        "D1_Base",
        "B3",
        model,
        "Kruskal-Wallis",
        "base_top5_by_band",
        H,
        p,
        eta_squared(groups_t5),
        "eta_squared",
    )

    # B4: Mean correct prob differs by band
    groups_mcp = [
        md[md["band"] == b]["base_mean_correct_prob"].dropna().values for b in BANDS
    ]
    H, p = safe_kruskal(*groups_mcp)
    acc.add_test(
        "D1_Base",
        "B4",
        model,
        "Kruskal-Wallis",
        "base_mcp_by_band",
        H,
        p,
        eta_squared(groups_mcp),
        "eta_squared",
    )

## 3. Domain 2: circuit band effects (C1-C4)

C1: KW on `circuit_accuracy`. C2: Jonckheere-Terpstra trend. C3, C4: KW on `retention_ratio` and `size_fraction`.

In [4]:
for model in MODELS:
    md = df_circuit[df_circuit["model"] == model]
    n_per_group = min(len(md[md["band"] == b]) for b in BANDS)
    # With n=3 per group (3 draws per band), Kruskal-Wallis has limited power.
    # Null results should be interpreted as inconclusive, not evidence of no effect.
    power_note = (
        f"n={n_per_group}/group; limited power for band discrimination"
        if n_per_group <= 5
        else None
    )

    # C1
    groups = [md[md["band"] == b]["circuit_accuracy"].dropna().values for b in BANDS]
    H, p = safe_kruskal(*groups)
    acc.add_test(
        "D2_Circuit",
        "C1",
        model,
        "Kruskal-Wallis",
        "circuit_acc_by_band",
        H,
        p,
        eta_squared(groups),
        "eta_squared",
        power_note=power_note,
    )

    # C2
    freq_groups = [
        md[md["band"] == b]["circuit_accuracy"].dropna().values for b in FREQUENCY_BANDS
    ]
    Z, p_jt = jonckheere_terpstra(freq_groups, alternative="increasing")
    acc.add_test(
        "D2_Circuit",
        "C2",
        model,
        "Jonckheere-Terpstra",
        "circuit_acc_trend",
        Z,
        p_jt,
        Z,
        "Z",
        power_note=power_note,
    )

    # C3
    groups_ret = [md[md["band"] == b]["retention_ratio"].dropna().values for b in BANDS]
    H, p = safe_kruskal(*groups_ret)
    acc.add_test(
        "D2_Circuit",
        "C3",
        model,
        "Kruskal-Wallis",
        "retention_by_band",
        H,
        p,
        eta_squared(groups_ret),
        "eta_squared",
        power_note=power_note,
    )

    # C4
    groups_size = [md[md["band"] == b]["size_fraction"].dropna().values for b in BANDS]
    H, p = safe_kruskal(*groups_size)
    acc.add_test(
        "D2_Circuit",
        "C4",
        model,
        "Kruskal-Wallis",
        "size_by_band",
        H,
        p,
        eta_squared(groups_size),
        "eta_squared",
        power_note=power_note,
    )

n_d2 = len([t for t in acc.tests if t["domain"] == "D2_Circuit"])
print(
    f"  Note: n={n_per_group}/group (draws per band). "
    f"D2 null results reflect limited power, not absence of effect."
)


  Note: n=3/group (draws per band). D2 null results reflect limited power, not absence of effect.


## 4. Domain 3: same vs cross-band generalization (G1-G4)

G1: one-sided MW for same > cross overall. G2: same > cross per model. G3: KW on cross-band accuracy by training band. G4: KW on the generalization gap by model.

In [5]:
# G1: Same > Cross overall
same_acc = (
    df_transfer[df_transfer["same_band"] == True]["circuit_accuracy"].dropna().values
)
cross_acc = (
    df_transfer[df_transfer["same_band"] == False]["circuit_accuracy"].dropna().values
)
U, p = safe_mannwhitneyu(same_acc, cross_acc, alternative="greater")
ci_l, ci_u, diff = bootstrap_ci_diff(same_acc, cross_acc)
acc.add_test(
    "D3_Generalization",
    "G1",
    "all",
    "Mann-Whitney",
    "same_vs_cross",
    U,
    p,
    rank_biserial(same_acc, cross_acc),
    "rank_biserial",
    n1=len(same_acc),
    n2=len(cross_acc),
    ci_lower=ci_l,
    ci_upper=ci_u,
)

# G2: Same > Cross per model
for model in MODELS:
    mt = df_transfer[df_transfer["model"] == model]
    same = mt[mt["same_band"] == True]["circuit_accuracy"].dropna().values
    cross = mt[mt["same_band"] == False]["circuit_accuracy"].dropna().values
    U, p = safe_mannwhitneyu(same, cross, alternative="greater")
    acc.add_test(
        "D3_Generalization",
        "G2",
        model,
        "Mann-Whitney",
        "same_vs_cross",
        U,
        p,
        rank_biserial(same, cross),
        "rank_biserial",
        n1=len(same),
        n2=len(cross),
    )

# G3: Cross-band accuracy differs by training band (per model)
cross_data = df_transfer[df_transfer["same_band"] == False]
for model in MODELS:
    mt = cross_data[cross_data["model"] == model]
    groups = [
        mt[mt["train_band"] == b]["circuit_accuracy"].dropna().values for b in BANDS
    ]
    H, p = safe_kruskal(*groups)
    acc.add_test(
        "D3_Generalization",
        "G3",
        model,
        "Kruskal-Wallis",
        "cross_acc_by_train_band",
        H,
        p,
        eta_squared(groups),
        "eta_squared",
    )

# G4: Generalization gap differs by model
gap_dfs = []
for model in MODELS:
    gap_dfs.append(compute_generalization_gap(df_transfer, model))
df_gaps = pd.concat(gap_dfs, ignore_index=True)
gap_groups = [df_gaps[df_gaps["model"] == m]["gap"].dropna().values for m in MODELS]
H, p = safe_kruskal(*gap_groups)
acc.add_test(
    "D3_Generalization",
    "G4",
    "all",
    "Kruskal-Wallis",
    "gap_by_model",
    H,
    p,
    eta_squared(gap_groups),
    "eta_squared",
)

## 5. Domain 4: asymmetric transfer (AS1-AS4)

AS1: two-sided MW for LF->HF vs HF->LF overall. AS2: per-model MW. AS3: KW on asymmetry magnitude by model. AS4: pairwise band asymmetry MW.

In [6]:
# AS1: Overall two-sided test
lf_to_hf = (
    df_transfer[
        (df_transfer["train_band"].isin(LOW_FREQ_BANDS))
        & (df_transfer["test_band"].isin(HIGH_FREQ_BANDS))
    ]["circuit_accuracy"]
    .dropna()
    .values
)

hf_to_lf = (
    df_transfer[
        (df_transfer["train_band"].isin(HIGH_FREQ_BANDS))
        & (df_transfer["test_band"].isin(LOW_FREQ_BANDS))
    ]["circuit_accuracy"]
    .dropna()
    .values
)

U, p = safe_mannwhitneyu(lf_to_hf, hf_to_lf, alternative="two-sided")
ci_l, ci_u, diff = bootstrap_ci_diff(lf_to_hf, hf_to_lf)
acc.add_test(
    "D4_Asymmetry",
    "AS1",
    "all",
    "Mann-Whitney",
    "LF_HF_vs_HF_LF_twosided",
    U,
    p,
    rank_biserial(lf_to_hf, hf_to_lf),
    "rank_biserial",
    n1=len(lf_to_hf),
    n2=len(hf_to_lf),
    ci_lower=ci_l,
    ci_upper=ci_u,
)

# Determine direction and test one-sided
if np.mean(lf_to_hf) > np.mean(hf_to_lf):
    U2, p2 = safe_mannwhitneyu(lf_to_hf, hf_to_lf, alternative="greater")
    direction = "LF_HF_gt_HF_LF"
else:
    U2, p2 = safe_mannwhitneyu(hf_to_lf, lf_to_hf, alternative="greater")
    direction = "HF_LF_gt_LF_HF"
acc.add_test(
    "D4_Asymmetry",
    "AS1_directed",
    "all",
    "Mann-Whitney",
    direction,
    U2,
    p2,
    rank_biserial(lf_to_hf, hf_to_lf),
    "rank_biserial",
)

print(
    f"Overall: LF->HF mean={np.mean(lf_to_hf):.4f}, HF->LF mean={np.mean(hf_to_lf):.4f}"
)
print(f"Direction: {direction}, p={p2:.4f}")

Overall: LF->HF mean=0.8321, HF->LF mean=0.7246
Direction: LF_HF_gt_HF_LF, p=0.0000


In [7]:
# AS2: Per-model asymmetry
for model in MODELS:
    mt = df_transfer[df_transfer["model"] == model]
    lf_hf = (
        mt[
            (mt["train_band"].isin(LOW_FREQ_BANDS))
            & (mt["test_band"].isin(HIGH_FREQ_BANDS))
        ]["circuit_accuracy"]
        .dropna()
        .values
    )
    hf_lf = (
        mt[
            (mt["train_band"].isin(HIGH_FREQ_BANDS))
            & (mt["test_band"].isin(LOW_FREQ_BANDS))
        ]["circuit_accuracy"]
        .dropna()
        .values
    )
    U, p = safe_mannwhitneyu(lf_hf, hf_lf, alternative="two-sided")
    acc.add_test(
        "D4_Asymmetry",
        "AS2",
        model,
        "Mann-Whitney",
        "LF_HF_vs_HF_LF",
        U,
        p,
        rank_biserial(lf_hf, hf_lf),
        "rank_biserial",
        n1=len(lf_hf),
        n2=len(hf_lf),
    )
    print(
        f"{model}: LF->HF={np.mean(lf_hf):.4f}, HF->LF={np.mean(hf_lf):.4f}, p={p:.4f}"
    )

# AS3: Asymmetry magnitude differs by model
asymmetry_per_draw = []
for model in MODELS:
    for draw in DRAWS:
        mt = df_transfer[
            (df_transfer["model"] == model) & (df_transfer["draw"] == draw)
        ]
        lf_hf = mt[
            (mt["train_band"].isin(LOW_FREQ_BANDS))
            & (mt["test_band"].isin(HIGH_FREQ_BANDS))
        ]["circuit_accuracy"].mean()
        hf_lf = mt[
            (mt["train_band"].isin(HIGH_FREQ_BANDS))
            & (mt["test_band"].isin(LOW_FREQ_BANDS))
        ]["circuit_accuracy"].mean()
        asymmetry_per_draw.append(
            {
                "model": model,
                "draw": draw,
                "lf_to_hf": lf_hf,
                "hf_to_lf": hf_lf,
                "asymmetry": lf_hf - hf_lf,
            }
        )
df_asymm = pd.DataFrame(asymmetry_per_draw)
model_groups = [
    df_asymm[df_asymm["model"] == m]["asymmetry"].dropna().values for m in MODELS
]
H, p = safe_kruskal(*model_groups)
acc.add_test(
    "D4_Asymmetry",
    "AS3",
    "all",
    "Kruskal-Wallis",
    "asymmetry_by_model",
    H,
    p,
    eta_squared(model_groups),
    "eta_squared",
)

# AS4: Pairwise band asymmetry
for lb in LOW_FREQ_BANDS:
    for hb in HIGH_FREQ_BANDS:
        a2b = (
            df_transfer[
                (df_transfer["train_band"] == lb) & (df_transfer["test_band"] == hb)
            ]["circuit_accuracy"]
            .dropna()
            .values
        )
        b2a = (
            df_transfer[
                (df_transfer["train_band"] == hb) & (df_transfer["test_band"] == lb)
            ]["circuit_accuracy"]
            .dropna()
            .values
        )
        U, p = safe_mannwhitneyu(a2b, b2a, alternative="two-sided")
        acc.add_test(
            "D4_Asymmetry",
            "AS4",
            "all",
            "Mann-Whitney",
            f"{lb}_to_{hb}_vs_{hb}_to_{lb}",
            U,
            p,
            rank_biserial(a2b, b2a),
            "rank_biserial",
            n1=len(a2b),
            n2=len(b2a),
        )

pythia-70m: LF->HF=0.4315, HF->LF=0.2933, p=0.0003
pythia-160m: LF->HF=0.9181, HF->LF=0.8615, p=0.0006
pythia-410m: LF->HF=0.9648, HF->LF=0.9174, p=0.0078
pythia-1b: LF->HF=0.9252, HF->LF=0.8341, p=0.0020
pythia-1.4b: LF->HF=0.9207, HF->LF=0.7167, p=0.0001



### D4 extension: pairwise band asymmetry (AS5-AS7)

Tests all 6 directed band pairs. AS5: per-pair MW two-sided, with overall + per-model breakdowns. AS6: Spearman of |asymmetry| vs frequency distance. AS7: binomial sign test that lower-frequency consistently transfers better to higher-frequency.

In [8]:
from scipy.stats import binomtest

# --- AS5: All pairwise directed comparisons ---
# For each undirected pair among FREQUENCY_BANDS, test transfer(A->B) vs transfer(B->A)
pairwise_results = []
n_as5 = 0

for i, band_a in enumerate(FREQUENCY_BANDS):
    for j, band_b in enumerate(FREQUENCY_BANDS):
        if i >= j:
            continue

        freq_dist = abs(FREQUENCY_RANK[band_a] - FREQUENCY_RANK[band_b])

        # Overall (all models pooled)
        a_to_b = (
            df_transfer[
                (df_transfer["train_band"] == band_a)
                & (df_transfer["test_band"] == band_b)
            ]["circuit_accuracy"]
            .dropna()
            .values
        )

        b_to_a = (
            df_transfer[
                (df_transfer["train_band"] == band_b)
                & (df_transfer["test_band"] == band_a)
            ]["circuit_accuracy"]
            .dropna()
            .values
        )

        U, p = safe_mannwhitneyu(a_to_b, b_to_a, alternative="two-sided")
        rb = rank_biserial(a_to_b, b_to_a)
        ci_l, ci_u, diff = bootstrap_ci_diff(a_to_b, b_to_a)
        acc.add_test(
            "D4_Asymmetry",
            "AS5",
            "all",
            "Mann-Whitney",
            f"{band_a}_to_{band_b}_vs_{band_b}_to_{band_a}",
            U,
            p,
            rb,
            "rank_biserial",
            n1=len(a_to_b),
            n2=len(b_to_a),
            ci_lower=ci_l,
            ci_upper=ci_u,
        )
        n_as5 += 1

        pairwise_results.append(
            {
                "band_a": band_a,
                "band_b": band_b,
                "a_to_b_mean": np.mean(a_to_b),
                "b_to_a_mean": np.mean(b_to_a),
                "asymmetry": np.mean(a_to_b) - np.mean(b_to_a),
                "abs_asymmetry": abs(np.mean(a_to_b) - np.mean(b_to_a)),
                "freq_distance": freq_dist,
                "p_value": p,
                "rank_biserial": rb,
            }
        )

        # Per-model
        # Note: With n=3 draws per condition, per-model Mann-Whitney has n1=n2=3.
        # Minimum achievable two-sided p-value is 0.10 (exact distribution has
        # only 20 permutations), so these tests CANNOT reach alpha=0.05.
        # They are included for completeness but should be interpreted with caution.
        for model in MODELS:
            mt = df_transfer[df_transfer["model"] == model]
            ma2b = (
                mt[(mt["train_band"] == band_a) & (mt["test_band"] == band_b)][
                    "circuit_accuracy"
                ]
                .dropna()
                .values
            )
            mb2a = (
                mt[(mt["train_band"] == band_b) & (mt["test_band"] == band_a)][
                    "circuit_accuracy"
                ]
                .dropna()
                .values
            )
            U, p = safe_mannwhitneyu(ma2b, mb2a, alternative="two-sided")
            n1, n2 = len(ma2b), len(mb2a)
            per_model_note = None
            if min(n1, n2) <= 3:
                per_model_note = (
                    f"min_achievable_p=0.10 (n1={n1},n2={n2}); "
                    f"cannot reach alpha={ALPHA}"
                )
            acc.add_test(
                "D4_Asymmetry",
                "AS5",
                model,
                "Mann-Whitney",
                f"{band_a}_to_{band_b}_vs_{band_b}_to_{band_a}",
                U,
                p,
                rank_biserial(ma2b, mb2a),
                "rank_biserial",
                n1=n1,
                n2=n2,
                power_note=per_model_note,
            )
            n_as5 += 1

df_pairwise = pd.DataFrame(pairwise_results)
print(
    f"  Note: Per-model tests have n1=n2=3 (min two-sided p=0.10, cannot reach alpha={ALPHA})"
)
print(f"\nPairwise asymmetry summary:")
for _, row in df_pairwise.iterrows():
    direction = "+" if row["asymmetry"] > 0 else "-"
    print(
        f"  {row['band_a']}->{row['band_b']}: {row['a_to_b_mean']:.4f} vs "
        f"{row['band_b']}->{row['band_a']}: {row['b_to_a_mean']:.4f} "
        f"(asym={row['asymmetry']:+.4f}, dist={row['freq_distance']}, p={row['p_value']:.4f})"
    )

# --- AS6: Asymmetry scales with frequency distance ---
rho, p = safe_spearmanr(df_pairwise["freq_distance"], df_pairwise["abs_asymmetry"])
acc.add_test(
    "D4_Asymmetry",
    "AS6",
    "all",
    "Spearman",
    "abs_asymmetry_vs_freq_distance",
    rho,
    p,
    rho,
    "spearman_rho",
    n1=len(df_pairwise),
)
print(f"\nAS6: |asymmetry| vs freq_distance: rho={rho:.4f}, p={p:.4f}")

# --- AS7: Direction consistency ---
# For pairs where rank(A) < rank(B), does A->B always > B->A?
# (lower-freq circuit transfers better to higher-freq data)
n_positive = (df_pairwise["asymmetry"] > 0).sum()
n_pairs = len(df_pairwise)
binom = binomtest(n_positive, n_pairs, p=0.5, alternative="greater")
acc.add_test(
    "D4_Asymmetry",
    "AS7",
    "all",
    "Binomial",
    "direction_consistency",
    n_positive,
    binom.pvalue,
    n_positive / n_pairs,
    "proportion",
    n1=n_pairs,
)
print(
    f"AS7: {n_positive}/{n_pairs} pairs show expected direction, p={binom.pvalue:.4f}"
)

# Save pairwise results
df_pairwise.to_csv(ANALYSIS_DIR / "pairwise_band_asymmetry.csv", index=False)
print(f"\nSaved: pairwise_band_asymmetry.csv")

print(
    f"\nD4 total tests: {len([t for t in acc.tests if t['domain'] == 'D4_Asymmetry'])}"
)


  Note: Per-model tests have n1=n2=3 (min two-sided p=0.10, cannot reach alpha=0.05)

Pairwise asymmetry summary:
  low->medium: 0.8107 vs medium->low: 0.7493 (asym=+0.0613, dist=1, p=0.0379)
  low->high: 0.8249 vs high->low: 0.7197 (asym=+0.1052, dist=2, p=0.0042)
  low->very_high: 0.8302 vs very_high->low: 0.6418 (asym=+0.1884, dist=3, p=0.0019)
  medium->high: 0.8323 vs high->medium: 0.7932 (asym=+0.0391, dist=1, p=0.1054)
  medium->very_high: 0.8409 vs very_high->medium: 0.7437 (asym=+0.0972, dist=2, p=0.0325)
  high->very_high: 0.8536 vs very_high->high: 0.8172 (asym=+0.0364, dist=1, p=0.4932)

AS6: |asymmetry| vs freq_distance: rho=0.9258, p=0.0080
AS7: 6/6 pairs show expected direction, p=0.0156

Saved: pairwise_band_asymmetry.csv

D4 total tests: 50


## 6. Domain 5: frequency distance (FD1-FD2)

FD1: Spearman of `circuit_accuracy` vs `freq_distance` overall. FD2: same per model.

In [9]:
cross_data = df_transfer[
    (df_transfer["same_band"] == False) & df_transfer["freq_distance"].notna()
]

# FD1: Overall
rho, p = safe_spearmanr(cross_data["freq_distance"], cross_data["circuit_accuracy"])
acc.add_test(
    "D5_Distance",
    "FD1",
    "all",
    "Spearman",
    "distance_vs_accuracy",
    rho,
    p,
    rho,
    "spearman_rho",
    n1=len(cross_data),
)
print(f"Overall: rho={rho:.4f}, p={p:.4f}")

# FD2: Per model
for model in MODELS:
    mt = cross_data[cross_data["model"] == model]
    rho, p = safe_spearmanr(mt["freq_distance"], mt["circuit_accuracy"])
    acc.add_test(
        "D5_Distance",
        "FD2",
        model,
        "Spearman",
        "distance_vs_accuracy",
        rho,
        p,
        rho,
        "spearman_rho",
        n1=len(mt),
    )
    print(f"{model}: rho={rho:.4f}, p={p:.4f}")

Overall: rho=-0.2107, p=0.0045
pythia-70m: rho=-0.0723, p=0.6750
pythia-160m: rho=-0.3164, p=0.0601
pythia-410m: rho=-0.4959, p=0.0021
pythia-1b: rho=-0.5914, p=0.0001
pythia-1.4b: rho=-0.2509, p=0.1399



## 7. Domain 6: model scaling (M1-M5)

M1: KW on `circuit_accuracy` across models. M2-M4: Spearman of `circuit_accuracy`, `size_fraction`, and `retention_ratio` vs model size. M5: KW on the generalization gap.

In [10]:
df_c = df_circuit.copy()
df_c["model_size"] = df_c["model"].map(MODEL_CAPACITY)

# M1
model_groups = [
    df_circuit[df_circuit["model"] == m]["circuit_accuracy"].dropna().values
    for m in MODELS
]
H, p = safe_kruskal(*model_groups)
acc.add_test(
    "D6_Scaling",
    "M1",
    "all",
    "Kruskal-Wallis",
    "circuit_acc_by_model",
    H,
    p,
    eta_squared(model_groups),
    "eta_squared",
)

# M2
rho, p = safe_spearmanr(df_c["model_size"], df_c["circuit_accuracy"])
acc.add_test(
    "D6_Scaling",
    "M2",
    "all",
    "Spearman",
    "accuracy_vs_size",
    rho,
    p,
    rho,
    "spearman_rho",
)

# M3
rho, p = safe_spearmanr(df_c["model_size"], df_c["size_fraction"])
acc.add_test(
    "D6_Scaling",
    "M3",
    "all",
    "Spearman",
    "size_frac_vs_model_size",
    rho,
    p,
    rho,
    "spearman_rho",
)

# M4
rho, p = safe_spearmanr(df_c["model_size"], df_c["retention_ratio"])
acc.add_test(
    "D6_Scaling",
    "M4",
    "all",
    "Spearman",
    "retention_vs_model_size",
    rho,
    p,
    rho,
    "spearman_rho",
)

# M5
gap_groups = [df_gaps[df_gaps["model"] == m]["gap"].dropna().values for m in MODELS]
H, p = safe_kruskal(*gap_groups)
acc.add_test(
    "D6_Scaling",
    "M5",
    "all",
    "Kruskal-Wallis",
    "gap_by_model",
    H,
    p,
    eta_squared(gap_groups),
    "eta_squared",
)

## 8. Domain 7: control vs frequency bands (CT1-CT2)

CT1: MW of control vs frequency-band-average circuit accuracy. CT2: MW of size fraction.

In [11]:
for model in MODELS:
    md = df_circuit[df_circuit["model"] == model]
    ctrl = md[md["band"] == "control"]
    freq = md[md["band"].isin(FREQUENCY_BANDS)]

    # CT1
    U, p = safe_mannwhitneyu(
        ctrl["circuit_accuracy"].values,
        freq["circuit_accuracy"].values,
        alternative="two-sided",
    )
    acc.add_test(
        "D7_Control",
        "CT1",
        model,
        "Mann-Whitney",
        "control_vs_freq_accuracy",
        U,
        p,
        rank_biserial(ctrl["circuit_accuracy"].values, freq["circuit_accuracy"].values),
        "rank_biserial",
        n1=len(ctrl),
        n2=len(freq),
    )

    # CT2
    U, p = safe_mannwhitneyu(
        ctrl["size_fraction"].values,
        freq["size_fraction"].values,
        alternative="two-sided",
    )
    acc.add_test(
        "D7_Control",
        "CT2",
        model,
        "Mann-Whitney",
        "control_vs_freq_size",
        U,
        p,
        rank_biserial(ctrl["size_fraction"].values, freq["size_fraction"].values),
        "rank_biserial",
        n1=len(ctrl),
        n2=len(freq),
    )

## 9. Domain 8: completeness (CP1-CP2)

CP1: paired Wilcoxon of circuit vs ablation accuracy. CP2: KW on completeness by band.

In [12]:
for model in MODELS:
    md = df_circuit[df_circuit["model"] == model]

    # CP1
    diff = md["circuit_accuracy"].values - md["ablation_accuracy"].values
    W, p = safe_wilcoxon(diff, alternative="greater")
    acc.add_test(
        "D8_Completeness",
        "CP1",
        model,
        "Wilcoxon",
        "circuit_gt_ablation",
        W,
        p,
        cohens_d_paired(diff),
        "cohens_d",
        n1=len(diff),
    )

    # CP2
    groups = [md[md["band"] == b]["completeness"].dropna().values for b in BANDS]
    H, p = safe_kruskal(*groups)
    acc.add_test(
        "D8_Completeness",
        "CP2",
        model,
        "Kruskal-Wallis",
        "completeness_by_band",
        H,
        p,
        eta_squared(groups),
        "eta_squared",
    )

## 10. Domain 9: variance decomposition (V1-V3)

Two-way ANOVA on `circuit_accuracy` for band x draw (V1), model x band (V2), and the model x band interaction (V3).

In [13]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

df_anova = df_circuit[["model", "band", "draw", "circuit_accuracy"]].dropna().copy()

# Fit single unified model with all factors
# (ensures all eta-squared values share the same denominator and sum to 1.0)
model_full = ols(
    "circuit_accuracy ~ C(model) + C(band) + C(draw) + C(model):C(band)", data=df_anova
).fit()
anova_full = sm.stats.anova_lm(model_full, typ=2)
ss_total = anova_full["sum_sq"].sum()

# Compute all eta-squared from the unified model
model_eta2 = anova_full.loc["C(model)", "sum_sq"] / ss_total
band_eta2 = anova_full.loc["C(band)", "sum_sq"] / ss_total
draw_eta2 = anova_full.loc["C(draw)", "sum_sq"] / ss_total
interaction_eta2 = anova_full.loc["C(model):C(band)", "sum_sq"] / ss_total
residual_eta2 = anova_full.loc["Residual", "sum_sq"] / ss_total

# V1: Band effect
acc.add_test(
    "D9_Variance",
    "V1",
    "all",
    "ANOVA (full model)",
    "band_effect",
    anova_full.loc["C(band)", "F"],
    anova_full.loc["C(band)", "PR(>F)"],
    band_eta2,
    "eta_squared",
    band_eta2=float(band_eta2),
    draw_eta2=float(draw_eta2),
)
print(
    f"V1 Band effect: eta2={band_eta2:.4f}, F={anova_full.loc['C(band)', 'F']:.2f}, "
    f"p={anova_full.loc['C(band)', 'PR(>F)']:.4f}"
)

# V2: Model effect
acc.add_test(
    "D9_Variance",
    "V2",
    "all",
    "ANOVA (full model)",
    "model_effect",
    anova_full.loc["C(model)", "F"],
    anova_full.loc["C(model)", "PR(>F)"],
    model_eta2,
    "eta_squared",
    model_eta2=float(model_eta2),
    band_eta2=float(band_eta2),
)
print(
    f"V2 Model effect: eta2={model_eta2:.4f}, F={anova_full.loc['C(model)', 'F']:.2f}, "
    f"p={anova_full.loc['C(model)', 'PR(>F)']:.4f}"
)

# V3: Model x Band interaction
acc.add_test(
    "D9_Variance",
    "V3",
    "all",
    "ANOVA (full model)",
    "model_x_band_interaction",
    anova_full.loc["C(model):C(band)", "F"],
    anova_full.loc["C(model):C(band)", "PR(>F)"],
    interaction_eta2,
    "eta_squared",
)
print(
    f"V3 Interaction: eta2={interaction_eta2:.4f}, F={anova_full.loc['C(model):C(band)', 'F']:.2f}, "
    f"p={anova_full.loc['C(model):C(band)', 'PR(>F)']:.4f}"
)

print(f"\nDraw effect: eta2={draw_eta2:.4f}")
print(f"Residual: eta2={residual_eta2:.4f}")
print(
    f"Sum of all eta2: {model_eta2 + band_eta2 + draw_eta2 + interaction_eta2 + residual_eta2:.4f}"
)

# Save decomposition (all from single model, sums to 1.0)
var_decomp = pd.DataFrame(
    {
        "factor": ["model", "band", "draw", "model:band", "residual"],
        "eta_squared": [
            float(model_eta2),
            float(band_eta2),
            float(draw_eta2),
            float(interaction_eta2),
            float(residual_eta2),
        ],
    }
)
var_decomp.to_csv(ANALYSIS_DIR / "variance_decomposition.csv", index=False)
print(f"\nSaved: variance_decomposition.csv")

V1 Band effect: eta2=0.0338, F=55.31, p=0.0000
V2 Model effect: eta2=0.9227, F=1509.64, p=0.0000
V3 Interaction: eta2=0.0354, F=14.49, p=0.0000

Draw effect: eta2=0.0007
Residual: eta2=0.0073
Sum of all eta2: 1.0000

Saved: variance_decomposition.csv


## 11. Domain 10: control as average circuit (CT3-CT6)

The control band uses frequency-weighted random sampling from the full vocabulary. If pooling produces an "average circuit", the control should be functionally average across bands, never best on any single band, and structurally equidistant from frequency-specific circuits. CT3: one-sided MW that control transfer variance < frequency-specific. CT4: binomial that control is never best-performing on any test band. CT5: descriptive (CV) of control's Jaccard similarity to bands. CT6: descriptive Jaccard difference vs cross-spectrum pairs.

In [14]:
# --- CT3: Control circuit transfer variance < frequency-specific circuits ---
# Control should have a flatter transfer profile (lower variance across test bands)
for model in MODELS:
    mt = df_transfer[df_transfer["model"] == model]

    # Compute per-draw transfer row variance for control vs frequency bands
    ctrl_draw_vars = []
    freq_draw_vars = []
    for draw in DRAWS:
        # Control circuit's accuracy across all 5 test bands for this draw
        ctrl_vals = mt[(mt["train_band"] == "control") & (mt["draw"] == draw)][
            "circuit_accuracy"
        ].values
        if len(ctrl_vals) > 1:
            ctrl_draw_vars.append(np.var(ctrl_vals))

        # Each frequency-band circuit's accuracy across all 5 test bands
        for band in FREQUENCY_BANDS:
            freq_vals = mt[(mt["train_band"] == band) & (mt["draw"] == draw)][
                "circuit_accuracy"
            ].values
            if len(freq_vals) > 1:
                freq_draw_vars.append(np.var(freq_vals))

    ctrl_draw_vars = np.array(ctrl_draw_vars)
    freq_draw_vars = np.array(freq_draw_vars)

    if len(ctrl_draw_vars) > 0 and len(freq_draw_vars) > 0:
        U, p = safe_mannwhitneyu(ctrl_draw_vars, freq_draw_vars, alternative="less")
        rb = rank_biserial(ctrl_draw_vars, freq_draw_vars)
        acc.add_test(
            "D10_ControlAvg",
            "CT3",
            model,
            "Mann-Whitney",
            "ctrl_transfer_var_lt_freq",
            U,
            p,
            rb,
            "rank_biserial",
            n1=len(ctrl_draw_vars),
            n2=len(freq_draw_vars),
        )
        print(
            f"CT3 {model}: ctrl_var={np.mean(ctrl_draw_vars):.6f}, "
            f"freq_var={np.mean(freq_draw_vars):.6f}, p={p:.4f}"
        )

# --- CT4: Control circuit never best-performing on any test band ---
for model in MODELS:
    mt = df_transfer[df_transfer["model"] == model]
    n_bands_best = 0
    for test_band in BANDS:
        mean_accs = {}
        for train_band in BANDS:
            vals = mt[
                (mt["train_band"] == train_band) & (mt["test_band"] == test_band)
            ]["circuit_accuracy"]
            mean_accs[train_band] = vals.mean() if len(vals) > 0 else 0
        best_train = max(mean_accs, key=mean_accs.get)
        if best_train == "control":
            n_bands_best += 1

    binom = binomtest(n_bands_best, len(BANDS), p=1 / len(BANDS), alternative="greater")
    acc.add_test(
        "D10_ControlAvg",
        "CT4",
        model,
        "Binomial",
        "ctrl_best_on_n_bands",
        n_bands_best,
        binom.pvalue,
        n_bands_best / len(BANDS),
        "proportion",
        n1=len(BANDS),
    )
    print(
        f"CT4 {model}: control is best on {n_bands_best}/{len(BANDS)} test bands, "
        f"p={binom.pvalue:.4f}"
    )

# --- CT5 & CT6: Structural centrality from Phase 2 Jaccard data ---
PHASE2_ANALYSIS = Path("LSC_circuit_analysis/02_Phase_Structural/outputs/analysis")
jaccard_path = PHASE2_ANALYSIS / "band_jaccard.csv"

if jaccard_path.exists():
    df_jaccard = pd.read_csv(jaccard_path)
    print(f"\nLoaded Phase 2 Jaccard data: {df_jaccard.shape}")

    for model in MODELS:
        mj = df_jaccard[df_jaccard["model"] == model]

        # CT5: Control's Jaccard with each frequency band: is it uniform?
        ctrl_jaccards = []
        for band in FREQUENCY_BANDS:
            j_val = mj[
                ((mj["band_1"] == "control") & (mj["band_2"] == band))
                | ((mj["band_1"] == band) & (mj["band_2"] == "control"))
            ]["mean_jaccard"]
            if len(j_val) > 0:
                ctrl_jaccards.append(j_val.values[0])

        if len(ctrl_jaccards) >= 2:
            cv = (
                np.std(ctrl_jaccards) / np.mean(ctrl_jaccards)
                if np.mean(ctrl_jaccards) > 0
                else np.nan
            )
            acc.add_test(
                "D10_ControlAvg",
                "CT5",
                model,
                "Descriptive",
                "ctrl_jaccard_uniformity",
                cv,
                np.nan,
                cv,
                "coefficient_of_variation",
                n1=len(ctrl_jaccards),
            )
            print(
                f"CT5 {model}: Jaccard(ctrl, freq bands) = {ctrl_jaccards}, CV={cv:.4f}"
            )

        # CT6: Control mean Jaccard vs cross-spectrum Jaccard
        ctrl_mean_j = mj[(mj["band_1"] == "control") | (mj["band_2"] == "control")][
            "mean_jaccard"
        ].mean()

        cross_spectrum_j = []
        for b1, b2 in CROSS_SPECTRUM_PAIRS:
            val = mj[
                ((mj["band_1"] == b1) & (mj["band_2"] == b2))
                | ((mj["band_1"] == b2) & (mj["band_2"] == b1))
            ]["mean_jaccard"]
            if len(val) > 0:
                cross_spectrum_j.append(val.values[0])

        if cross_spectrum_j:
            cross_mean = np.mean(cross_spectrum_j)
            diff = ctrl_mean_j - cross_mean
            acc.add_test(
                "D10_ControlAvg",
                "CT6",
                model,
                "Descriptive",
                "ctrl_jaccard_gt_cross_spectrum",
                diff,
                np.nan,
                diff,
                "difference",
                n1=1,
            )
            print(
                f"CT6 {model}: ctrl_mean_J={ctrl_mean_j:.4f}, "
                f"cross_spectrum_J={cross_mean:.4f}, diff={diff:+.4f}"
            )
else:
    print(f"WARNING: Phase 2 Jaccard data not found at {jaccard_path}")

CT3 pythia-70m: ctrl_var=0.013297, freq_var=0.009867, p=0.7758
CT3 pythia-160m: ctrl_var=0.001553, freq_var=0.001183, p=0.8527
CT3 pythia-410m: ctrl_var=0.000794, freq_var=0.000763, p=0.8176
CT3 pythia-1b: ctrl_var=0.002419, freq_var=0.002422, p=0.9319
CT3 pythia-1.4b: ctrl_var=0.004770, freq_var=0.007203, p=0.5802
CT4 pythia-70m: control is best on 4/5 test bands, p=0.0067
CT4 pythia-160m: control is best on 1/5 test bands, p=0.6723
CT4 pythia-410m: control is best on 1/5 test bands, p=0.6723


CT4 pythia-1b: control is best on 1/5 test bands, p=0.6723


CT4 pythia-1.4b: control is best on 1/5 test bands, p=0.6723

Loaded Phase 2 Jaccard data: (100, 4)
CT5 pythia-70m: Jaccard(ctrl, freq bands) = [0.7267666640558113, 0.7398682300719115, 0.7774932826809255, 0.7886704625731809], CV=0.0338
CT6 pythia-70m: ctrl_mean_J=0.7594, cross_spectrum_J=0.7454, diff=+0.0140
CT5 pythia-160m: Jaccard(ctrl, freq bands) = [0.5590635483209955, 0.5533401069311022, 0.5543742527731204, 0.5682932348747672], CV=0.0106
CT6 pythia-160m: ctrl_mean_J=0.5622, cross_spectrum_J=0.5402, diff=+0.0220
CT5 pythia-410m: Jaccard(ctrl, freq bands) = [0.4228547071319675, 0.431964803286541, 0.4358867505427181, 0.4409720781080006], CV=0.0153
CT6 pythia-410m: ctrl_mean_J=0.4335, cross_spectrum_J=0.4176, diff=+0.0159
CT5 pythia-1b: Jaccard(ctrl, freq bands) = [0.4646744867244538, 0.4598281488112861, 0.4795350139284162, 0.489332907704707], CV=0.0248
CT6 pythia-1b: ctrl_mean_J=0.4745, cross_spectrum_J=0.4536, diff=+0.0210



## 12. Domain 11: random subcircuit baseline (RB1-RB3)

Validates ACDC edges by comparing each circuit's accuracy against K=100 random edge sets of the same size. Prerequisite: run `lsc_random_baseline.py` first to generate `random_baseline_results.json`. RB1: Wilcoxon signed-rank that real-circuit z-scores > 0 overall. RB2: same per model. RB3: binomial that proportion with percentile rank >= 95% exceeds chance.

In [15]:
# --- D11: Random Subcircuit Baseline (RB1-RB3) ---
rb_path = RANDOM_BASELINE_RESULTS
if rb_path.exists():
    with open(rb_path) as f:
        rb_data = json.load(f)
    df_rb = pd.DataFrame(rb_data["results"])
    print(
        f"Loaded random baseline data: {len(df_rb)} circuits, K={rb_data.get('K', '?')}"
    )

    # Cap infinite z-scores at Z_SCORE_CAP for statistical testing.
    # Inf occurs when std_random_accuracy = 0 (all K random circuits scored 0%).
    # These represent the strongest possible evidence, not missing data.
    Z_SCORE_CAP = 1000.0
    z_scores_raw = df_rb["z_score"].dropna().values
    n_inf = int(np.sum(np.isinf(z_scores_raw)))
    n_finite_large = (
        int(np.sum(np.abs(z_scores_raw[np.isfinite(z_scores_raw)]) > Z_SCORE_CAP))
        if np.any(np.isfinite(z_scores_raw))
        else 0
    )
    z_capped = np.clip(z_scores_raw, -Z_SCORE_CAP, Z_SCORE_CAP)
    print(
        f"Z-scores: {len(z_scores_raw)} total, {n_inf} infinite, "
        f"{n_finite_large} finite but >{Z_SCORE_CAP} (all capped at +/-{Z_SCORE_CAP})"
    )

    # RB1: Overall: real z-scores significantly > 0
    if len(z_capped) >= 3:
        W, p = safe_wilcoxon(z_capped, alternative="greater")
        acc.add_test(
            "D11_RandomBaseline",
            "RB1",
            "all",
            "Wilcoxon",
            "z_scores_gt_zero",
            W,
            p,
            cohens_d_paired(z_capped),
            "cohens_d",
            n1=len(z_capped),
            n_inf_capped=n_inf,
        )
        print(
            f"RB1: median z-score (capped)={np.median(z_capped):.2f}, "
            f"n={len(z_capped)}, p={p:.6f}"
        )

    # RB2: Per model
    for model in MODELS:
        model_mask = df_rb["model"] == model
        mz_raw = df_rb.loc[model_mask, "z_score"].dropna().values
        mz_capped = np.clip(mz_raw, -Z_SCORE_CAP, Z_SCORE_CAP)
        n_inf_model = int(np.sum(np.isinf(mz_raw)))
        if len(mz_capped) >= 3:
            W, p = safe_wilcoxon(mz_capped, alternative="greater")
            acc.add_test(
                "D11_RandomBaseline",
                "RB2",
                model,
                "Wilcoxon",
                "z_scores_gt_zero",
                W,
                p,
                cohens_d_paired(mz_capped),
                "cohens_d",
                n1=len(mz_capped),
                n_inf_capped=n_inf_model,
            )
            print(
                f"RB2 {model}: median z={np.median(mz_capped):.2f}, "
                f"n={len(mz_capped)} ({n_inf_model} were inf, capped), p={p:.6f}"
            )

    # RB3: Proportion with percentile >= 95
    n_above_95 = (df_rb["percentile_rank"] >= 95).sum()
    n_total_rb = len(df_rb)
    binom = binomtest(n_above_95, n_total_rb, p=0.05, alternative="greater")
    acc.add_test(
        "D11_RandomBaseline",
        "RB3",
        "all",
        "Binomial",
        "pct_above_95th_percentile",
        n_above_95,
        binom.pvalue,
        n_above_95 / n_total_rb,
        "proportion",
        n1=n_total_rb,
    )
    print(
        f"RB3: {n_above_95}/{n_total_rb} circuits above 95th percentile, p={binom.pvalue:.6f}"
    )

    # Save summary with both raw and capped z-scores
    rb_summary = df_rb[
        [
            "model",
            "band",
            "draw",
            "n_edges",
            "real_accuracy",
            "mean_random_accuracy",
            "std_random_accuracy",
            "z_score",
            "percentile_rank",
        ]
    ].copy()
    rb_summary["z_score_capped"] = np.clip(
        rb_summary["z_score"].values, -Z_SCORE_CAP, Z_SCORE_CAP
    )
    rb_summary.to_csv(ANALYSIS_DIR / "random_baseline_summary.csv", index=False)
    print(f"\nSaved: random_baseline_summary.csv")
else:
    print(f"WARNING: Random baseline results not found at {rb_path}")
    print("Run lsc_random_baseline.py first, then re-run this notebook.")

Loaded random baseline data: 60 circuits, K=100
Z-scores: 60 total, 46 infinite, 7 finite but >1000.0 (all capped at +/-1000.0)
RB1: median z-score (capped)=1000.00, n=60, p=0.000000
RB2 pythia-70m: median z=1000.00, n=15 (10 were inf, capped), p=0.000146
RB2 pythia-160m: median z=1000.00, n=15 (11 were inf, capped), p=0.000179
RB2 pythia-410m: median z=1000.00, n=15 (11 were inf, capped), p=0.000054
RB2 pythia-1b: median z=1000.00, n=15 (14 were inf, capped), p=0.000054
RB3: 60/60 circuits above 95th percentile, p=0.000000

Saved: random_baseline_summary.csv



## 11. Multiple Comparison Correction

In [16]:
df_tests = acc.to_dataframe()
df_tests = acc.apply_fdr_correction(df_tests)

print(f"Total tests: {len(df_tests)}")
print(
    f"Significant (uncorrected, alpha={ALPHA}): {df_tests['significant_uncorrected'].sum()}"
)
print(f"Significant (BH-FDR corrected): {df_tests['significant_bh'].sum()}")

Total tests: 160
Significant (uncorrected, alpha=0.05): 61
Significant (BH-FDR corrected): 38


## 12. Results Summary by Domain

In [17]:
for domain in df_tests["domain"].unique():
    dt = df_tests[df_tests["domain"] == domain]
    n_sig = dt["significant_bh"].sum()
    n_total = len(dt)
    print(f"\n{domain}: {n_sig}/{n_total} significant (BH-FDR)")
    sig = dt[dt["significant_bh"]]
    if len(sig) > 0:
        for _, row in sig.iterrows():
            print(
                f"  {row['hypothesis']} [{row['model']}]: "
                f"p={row['p_value']:.4f} -> p_bh={row['p_value_bh']:.4f}, "
                f"effect={row['effect_size']:.3f} ({row['interpretation']})"
            )


D1_Base: 4/20 significant (BH-FDR)
  B1 [pythia-70m]: p=0.0091 -> p_bh=0.0373, effect=0.950 (large)
  B3 [pythia-70m]: p=0.0107 -> p_bh=0.0429, effect=0.912 (large)
  B4 [pythia-70m]: p=0.0091 -> p_bh=0.0373, effect=0.950 (large)
  B4 [pythia-160m]: p=0.0091 -> p_bh=0.0373, effect=0.950 (large)

D2_Circuit: 0/20 significant (BH-FDR)

D3_Generalization: 1/12 significant (BH-FDR)
  G2 [pythia-1b]: p=0.0074 -> p_bh=0.0364, effect=0.409 (medium)

D4_Asymmetry: 12/50 significant (BH-FDR)
  AS1 [all]: p=0.0000 -> p_bh=0.0001, effect=0.494 (medium)
  AS1_directed [all]: p=0.0000 -> p_bh=0.0000, effect=0.494 (medium)
  AS2 [pythia-70m]: p=0.0003 -> p_bh=0.0024, effect=0.882 (large)
  AS2 [pythia-160m]: p=0.0006 -> p_bh=0.0038, effect=0.833 (large)
  AS2 [pythia-410m]: p=0.0078 -> p_bh=0.0369, effect=0.646 (large)
  AS2 [pythia-1b]: p=0.0020 -> p_bh=0.0120, effect=0.750 (large)
  AS2 [pythia-1.4b]: p=0.0001 -> p_bh=0.0011, effect=0.951 (large)
  AS4 [all]: p=0.0042 -> p_bh=0.0227, effect=0.618

## 13. Visualizations

### Effect sizes by domain

In [18]:
fig, ax = plt.subplots(figsize=(14, 6))
valid = df_tests[
    df_tests["effect_size"].notna() & (df_tests["effect_type"] != "Z")
].copy()
if len(valid) > 0:
    valid["abs_effect"] = valid["effect_size"].abs()
    sns.boxplot(data=valid, x="domain", y="abs_effect", ax=ax, palette="Set2")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_ylabel("|Effect Size|")
    ax.set_title("Effect Size Distribution by Domain", fontsize=14)
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="Medium threshold")
    ax.legend()
fig.tight_layout()
save_figure(fig, "11_effect_sizes_by_domain.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/11_effect_sizes_by_domain.png


### Significance rates by domain

In [19]:
fig, ax = plt.subplots(figsize=(12, 6))
sig_summary = (
    df_tests.groupby("domain")
    .agg(
        n_sig_uncorr=("significant_uncorrected", "sum"),
        n_sig_bh=("significant_bh", "sum"),
        n_total=("significant_bh", "count"),
    )
    .reset_index()
)
sig_summary["pct_sig_bh"] = sig_summary["n_sig_bh"] / sig_summary["n_total"] * 100

x = np.arange(len(sig_summary))
width = 0.35
ax.bar(
    x - width / 2, sig_summary["n_sig_uncorr"], width, label="Uncorrected", alpha=0.7
)
ax.bar(x + width / 2, sig_summary["n_sig_bh"], width, label="BH-FDR", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(sig_summary["domain"], rotation=45, ha="right")
ax.set_ylabel("Number of Significant Tests")
ax.set_title("Significant Tests by Domain", fontsize=14)
ax.legend()

# Add total counts
for i, row in sig_summary.iterrows():
    ax.text(
        i,
        max(row["n_sig_uncorr"], row["n_sig_bh"]) + 0.3,
        f"/{row['n_total']}",
        ha="center",
        fontsize=9,
        color="gray",
    )

fig.tight_layout()
save_figure(fig, "12_significance_rates.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/12_significance_rates.png


### P-value distribution

In [20]:
fig, ax = plt.subplots(figsize=(10, 6))
valid_pvals = df_tests["p_value"].dropna().values
ax.hist(valid_pvals, bins=20, edgecolor="black", alpha=0.7, color="steelblue")
ax.axvline(ALPHA, color="red", linestyle="--", linewidth=2, label=f"alpha={ALPHA}")
ax.set_xlabel("P-value")
ax.set_ylabel("Count")
ax.set_title(f"P-value Distribution ({len(valid_pvals)} tests)", fontsize=14)
ax.legend()
fig.tight_layout()
save_figure(fig, "13_pvalue_distribution.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/13_pvalue_distribution.png


### Variance decomposition

In [21]:
fig, ax = plt.subplots(figsize=(10, 6))
var_decomp_plot = var_decomp.copy()
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#7f7f7f"]
bars = ax.bar(
    var_decomp_plot["factor"],
    var_decomp_plot["eta_squared"],
    color=colors,
    alpha=0.8,
    edgecolor="black",
)
ax.set_ylabel("Proportion of Variance (eta-squared)")
ax.set_title("Variance Decomposition of Circuit Accuracy", fontsize=14)
ax.bar_label(bars, fmt="%.3f", padding=2)
fig.tight_layout()
save_figure(fig, "14_variance_decomposition.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/14_variance_decomposition.png


### Frequency distance vs transfer accuracy

In [22]:
cross_freq = df_transfer[
    (df_transfer["same_band"] == False) & df_transfer["freq_distance"].notna()
].copy()

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)
for i, model in enumerate(MODELS):
    ax = axes[i]
    mt = cross_freq[cross_freq["model"] == model]
    sns.boxplot(
        data=mt,
        x="freq_distance",
        y="circuit_accuracy",
        ax=ax,
        color="steelblue",
        width=0.5,
    )
    sns.stripplot(
        data=mt,
        x="freq_distance",
        y="circuit_accuracy",
        ax=ax,
        color="black",
        alpha=0.4,
        size=4,
    )
    ax.set_title(model, fontsize=13)
    ax.set_xlabel("Frequency Distance")
    if i == 0:
        ax.set_ylabel("Circuit Accuracy")
    else:
        ax.set_ylabel("")
fig.suptitle("Transfer Accuracy vs Frequency Distance", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "15_freq_distance_vs_transfer.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/15_freq_distance_vs_transfer.png


### Asymmetry per draw

In [23]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=df_asymm,
    x="model",
    y="asymmetry",
    ax=ax,
    palette=MODEL_COLORS,
    errorbar="sd",
    capsize=0.1,
)
sns.stripplot(
    data=df_asymm, x="model", y="asymmetry", ax=ax, color="black", alpha=0.6, size=8
)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_ylabel("Asymmetry (LF->HF - HF->LF)")
ax.set_title("Transfer Asymmetry by Model (with individual draws)", fontsize=14)
fig.tight_layout()
save_figure(fig, "16_asymmetry_per_draw.png")

  Saved: LSC_circuit_analysis/01_Phase_Functional/outputs/viz/16_asymmetry_per_draw.png


## 14. Export

In [24]:
df_tests.to_csv(ANALYSIS_DIR / "all_statistical_tests.csv", index=False)
print(f"Saved: all_statistical_tests.csv ({len(df_tests)} tests)")

domain_summary = (
    df_tests.groupby("domain")
    .agg(
        n_tests=("hypothesis", "count"),
        n_sig_uncorrected=("significant_uncorrected", "sum"),
        n_sig_bh=("significant_bh", "sum"),
        mean_effect=("effect_size", lambda x: x[x.notna()].mean()),
    )
    .reset_index()
)
domain_summary.to_csv(ANALYSIS_DIR / "summary_by_domain.csv", index=False)
print(f"Saved: summary_by_domain.csv")

df_asymm.to_csv(ANALYSIS_DIR / "asymmetry_per_draw.csv", index=False)
print(f"Saved: asymmetry_per_draw.csv")

Saved: all_statistical_tests.csv (160 tests)
Saved: summary_by_domain.csv
Saved: asymmetry_per_draw.csv


## Summary

In [25]:
print("=" * 80)
print("INFERENTIAL STATISTICS SUMMARY")
print("=" * 80)
print(f"\nTotal tests: {len(df_tests)}")
print(
    f"Significant (alpha={ALPHA}, uncorrected): {df_tests['significant_uncorrected'].sum()}"
)
print(f"Significant (BH-FDR corrected): {df_tests['significant_bh'].sum()}")
print()
print(acc.summary(df_tests))
print()
print("Key findings (BH-FDR significant):")
sig_tests = df_tests[df_tests["significant_bh"]].sort_values("p_value_bh")
for _, row in sig_tests.head(15).iterrows():
    print(
        f"  {row['domain']}/{row['hypothesis']} [{row['model']}]: "
        f"p_bh={row['p_value_bh']:.4f}, effect={row['effect_size']:.3f} "
        f"({row['effect_type']}: {row['interpretation']})"
    )

INFERENTIAL STATISTICS SUMMARY

Total tests: 160
Significant (alpha=0.05, uncorrected): 61
Significant (BH-FDR corrected): 38

D1_Base: 4/20 significant
D2_Circuit: 0/20 significant
D3_Generalization: 1/12 significant
D4_Asymmetry: 12/50 significant
D5_Distance: 3/6 significant
D6_Scaling: 3/5 significant
D7_Control: 0/10 significant
D8_Completeness: 5/10 significant
D9_Variance: 3/3 significant
D10_ControlAvg: 1/18 significant
D11_RandomBaseline: 6/6 significant

Key findings (BH-FDR significant):
  D11_RandomBaseline/RB3 [all]: p_bh=0.0000, effect=1.000 (proportion: N/A)
  D9_Variance/V2 [all]: p_bh=0.0000, effect=0.923 (eta_squared: large)
  D6_Scaling/M3 [all]: p_bh=0.0000, effect=-0.882 (spearman_rho: large)
  D9_Variance/V1 [all]: p_bh=0.0000, effect=0.034 (eta_squared: small)
  D11_RandomBaseline/RB1 [all]: p_bh=0.0000, effect=5.734 (cohens_d: large)
  D9_Variance/V3 [all]: p_bh=0.0000, effect=0.035 (eta_squared: small)
  D6_Scaling/M1 [all]: p_bh=0.0000, effect=0.793 (eta_squar

In [26]:
print("\nEXPORTED FILES")
print("=" * 80)
print("\nCSV files:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nVisualizations:")
for f in sorted(VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")


EXPORTED FILES

CSV files:
  all_statistical_tests.csv
  asymmetry_per_draw.csv
  asymmetry_summary.csv
  base_model_stats.csv
  bimodality_results.csv
  circuit_size_stats.csv
  completeness_stats.csv
  failure_analysis_summary.csv
  faithfulness_stats.csv
  full_circuit_data.csv
  full_transfer_data.csv
  generalization_gap_per_draw.csv
  generalization_gap_stats.csv
  master_summary.csv
  pairwise_band_asymmetry.csv
  per_example_robustness.csv
  random_baseline_summary.csv
  scaling_summary.csv
  summary_by_domain.csv
  variance_decomposition.csv

Visualizations:
  01_base_model_accuracy.png
  02_circuit_size_heatmap.png
  03_same_band_accuracy_heatmap.png
  04_retention_ratio_by_band.png
  05_cross_band_transfer_matrices.png
  06_asymmetric_transfer.png
  06b_pairwise_asymmetry.png
  06c_pairwise_asymmetry_distance.png
  07_three_way_comparison.png
  08_model_scaling.png
  09_accuracy_vs_sparsity.png
  10_kl_divergence_analysis.png
  11_effect_sizes_by_domain.png
  12_significanc